In [2]:
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain.tools import tool
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
PRODUCTS = {
    "wireless headphones": {"price": 79.99,  "rating": 4.6, "description": "Over-ear Bluetooth, 30-hr battery, active noise cancellation."},
    "smart watch":         {"price": 199.99, "rating": 4.3, "description": "Tracks heart rate and sleep. 5-day battery, water-resistant."},
    "mechanical keyboard": {"price": 129.00, "rating": 4.8, "description": "Tenkeyless, Cherry MX Brown switches, per-key RGB."},
    "laptop stand":        {"price": 34.99,  "rating": 4.5, "description": "Adjustable aluminium, fits 11-17 inch laptops, folds flat."},
}

@tool
def get_product(name: str) -> str:
    """Look up a product by name and return its price, rating, stock, and description."""
    p = PRODUCTS.get(name.lower())
    if not p:
        return f"Product not found. Available: {', '.join(PRODUCTS)}"
    return str(p)

In [4]:
# create llm 
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)
agent = create_agent(
    llm,
    tools=[get_product],
    system_prompt="You are a helpful assistant for an online electronics store.",
)


In [5]:
def ask(question: str) -> str:
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    print(result["messages"][-1].content)

In [6]:
ask("What is the price and rating of the smart watch?")

The smart watch is priced at **$199.99** and has a rating of **4.3 out of 5**.


In [7]:
REVIEWS = {
    "wireless headphones": {"reviews": 1262, "rating": 4.6},
    "smart watch":         {"reviews": 340,  "rating": 3.9},
    "mechanical keyboard": {"reviews": 67,   "rating": 4.8},
    "laptop stand":        {"reviews": 781,  "rating": 4.5},
}

@tool(description="Get the review and rating of a product")
def get_review(product_name: str) -> str:
    review = REVIEWS.get(product_name.lower())
    if review:
        return review
    return f"No reviews found for {product_name}."

In [8]:
productAgent = create_agent(
    llm,
    tools=[get_review, get_product],
    system_prompt="You are a helpful assistant for an online electronics store.",
)

def ask_product_agent(question: str) -> str:
    result = productAgent.invoke({"messages": [{"role": "user", "content": question}]})
    print(result["messages"][-1].content)
    return result["messages"][-1].content

In [9]:
ask_product_agent("How do people like smart watch?")

People seem to feel fairly positive about the smart watch overall. It has gathered **340 reviews** and holds an average **rating of 3.9 / 5**. This suggests that most owners are satisfied, though there’s still room for improvement in certain areas. If you’re considering buying one, the feedback indicates it’s generally well‑received, but you might want to read a few individual reviews to see if it meets your specific needs.


'People seem to feel fairly positive about the smart watch overall. It has gathered **340 reviews** and holds an average **rating of 3.9\u202f/\u202f5**. This suggests that most owners are satisfied, though there’s still room for improvement in certain areas. If you’re considering buying one, the feedback indicates it’s generally well‑received, but you might want to read a few individual reviews to see if it meets your specific needs.'

In [10]:
ask_product_agent("What is the price of the smart watch?")

The smart watch is priced at **$199.99**.


'The smart watch is priced at **$199.99**.'

In [11]:
ask_product_agent("What are review on this product")

Sure! Could you let me know which product you’d like to see the reviews for? Just give me the product name, and I’ll pull up the latest ratings and comments for you.


'Sure! Could you let me know which product you’d like to see the reviews for? Just give me the product name, and I’ll pull up the latest ratings and comments for you.'

In [19]:
## Lets try In-memory memory with the product agent
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()

productAgent = create_agent(
    llm,
    tools=[get_review, get_product],
    system_prompt="You are a helpful assistant for an online electronics store.",
    checkpointer=memory,
)

def ask_product_agent(question: str) -> str:
    config = {"configurable": {"thread_id": "user-1"}}
    result = productAgent.invoke({"messages": [{"role": "user", "content": question}]}, config=config)
    return result["messages"][-1].content

In [20]:
ask_product_agent("How is the price of the smart watch?")

'The smart watch is priced at **$199.99**.'

In [21]:
ask_product_agent("whats the rating for this?")

'The smart watch has a rating of **4.3 out of 5**.'